# Track 01 Starter

This scaffold loads the Track 01 data package and checks integrity before exploratory plotting. It does not select a contrast and does not compute a final result.


In [ ]:
from IPython.display import HTML

HTML("""<style>
body { font-family: 'DM Sans', 'Helvetica Neue', sans-serif; }
</style>""")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA

DATA_DIR = Path('../../data/track-01')
COUNTS_PATH = DATA_DIR / 'track01_counts_vt.csv'
METADATA_PATH = DATA_DIR / 'track01_metadata_vt.csv'

def load_if_available(path):
    if not path.exists() or path.stat().st_size == 0:
        print(f'{path.name} is pending HDAG release.')
        return None
    return pd.read_csv(path, index_col=0)

## Load the data

Both files come from the Track 01 data package. The count matrix has genes as rows and samples as columns. The metadata table has one row per sample.


In [ ]:
counts = load_if_available(COUNTS_PATH)
metadata = load_if_available(METADATA_PATH)

if counts is not None:
    print('counts:', counts.shape)
if metadata is not None:
    print('metadata:', metadata.shape)

## Integrity checks

Confirm the data are what they claim to be before any analysis. Do not silently fix values that fail a check. Report them in your interpretation memo.


In [ ]:
def check_integrity(counts, metadata):
    if counts is None or metadata is None:
        return
    sample_overlap = set(metadata.index) & set(counts.columns)
    print('samples in both files:', len(sample_overlap))
    print('counts are nonnegative:', (counts.values >= 0).all())
    print('counts are integral:', (counts.values % 1 == 0).all())
    print('duplicate genes:', counts.index.duplicated().sum())
    print('duplicate samples:', counts.columns.duplicated().sum())
    if 'group' in metadata.columns:
        print(metadata['group'].value_counts())

check_integrity(counts, metadata)

## Exploratory plots

Library-size normalize, log transform, then look at overall structure. These plots are exploratory only and do not constitute a result.


In [ ]:
def exploratory_plots(counts, metadata):
    if counts is None:
        return
    lib_size = counts.sum(axis=0)
    normalized = np.log1p(counts.div(lib_size, axis=1))
    pca = PCA(n_components=2)
    scores = pca.fit_transform(normalized.T)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(scores[:, 0], scores[:, 1])
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].set_title('PCA')
    dist = np.sqrt(((scores[:, None, :] - scores[None, :, :]) ** 2).sum(axis=2))
    im = axes[1].imshow(dist)
    axes[1].set_title('Sample distances')
    fig.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.show()
    if metadata is not None and 'group' in metadata.columns:
        print('Color points by metadata fields to inspect structure before choosing a contrast.')

exploratory_plots(counts, metadata)

## Next steps

- Choose one contrast from the challenge guide and defend it.
- Run differential expression on that contrast.
- Run one pathway-level analysis with the Hallmark gene sets from the fetch script.
- Localize three to six genes with the Zeisel browser and state one hypothesis with its limitations.
